# Testing Model
- This notebook runs the model in inference mode for demo purpose, i.e. testing

## Install dependencies

In [ ]:
!pip install matplotlib pandas sentence_transformers torch transformers
!python -m spacy download en_core_web_md

## Import dependencies

In [ ]:
from collections import defaultdict, Counter
from itertools import product
import math
import matplotlib.pyplot as plt
from nltk import ngrams
import numpy as np
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
import spacy
from spacy.symbols import ORTH, POS
import torch
from torch import nn
from torch.amp import autocast, GradScaler
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

print(f"Import is finished ...")

## Customised Classes

### AVDataset
- This class supports reading both training and testing data
    - Training data contains "label"
    - Testing data does not contain "label"
- #### Columns of ```csv_file_normal```
    - ```text_1```
    - ```text_2```
    - ```author_score```
    - ```style_similarity_score```
- #### Columns of ```csv_file_manual_stylometry```
    - ```0```
    - ```1```
    - ...
    - ```170```
    - Each column represents 1 feature and there are $171$ features

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class AVDataset(Dataset):
    def __init__(self, csv_file_normal: str, csv_file_manual_stylometry: str, text_1_column="text_1", text_2_column="text_2", author_score_column="author_score", style_similarity_score_column="style_similarity_score", separation_token="[SEP]"):
        # Load CSV
        self.df_normal = pd.read_csv(csv_file_normal)
        self.df_manual_stylomtry = pd.read_csv(csv_file_manual_stylometry)

        assert len(self.df_normal) == len(self.df_manual_stylomtry), f"Mismatch between lengths of dataframes: {len(self.df_normal)} != {len(self.df_manual_stylomtry)}"

        # Columns
        self.text_1 = self.df_normal[text_1_column].astype(str).tolist()
        self.text_2 = self.df_normal[text_2_column].astype(str).tolist()
        if author_score_column in self.df_normal.columns:
            # Non-testing
            self.author_score = self.df_normal[author_score_column].astype(float).tolist()
        else:
            # Testing
            self.author_score = None
        self.style_similarity_score = self.df_normal[style_similarity_score_column].astype(float).tolist()

        # Load the whole csv file
        self.manual_stylomtries = self.df_manual_stylomtry.values

        assert len(self.df_normal) == len(self.manual_stylomtries), f"Mismatch between lengths between self.df_normal and self.manual_stylometries: {len(self.df_normal)} != {len(self.manual_stylomtries)}"

        self.separation_token = separation_token

        # Sanity check
        if self.author_score is not None:
            # Non-testing
            assert len(self.text_1) == len(self.author_score) and len(self.text_2) == len(self.author_score) and len(self.style_similarity_score) == len(self.author_score), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)}, author_score={len(self.author_score)}, style_similarity_score={len(self.style_similarity_score)}"
        else:
            # Testing
            assert len(self.text_1) == len(self.text_2) and len(self.style_similarity_score) == len(self.text_2), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)} style_similarity_score={len(self.style_similarity_score)}"

    def __len__(self):
        return len(self.text_1)

    def __getitem__(self, idx):
        """
        batch_manual_stylometry_y is literally author_score, which is the label
        """
        # Return a tuple
        combined_text = (self.text_1[idx], self.text_2[idx])
        if self.author_score is not None:
            # Non-testing
            author_score = torch.tensor(self.author_score[idx], dtype=torch.float)
        else:
            author_score = None
        style_similarity_score = torch.tensor(
            self.style_similarity_score[idx], dtype=torch.float)
        manual_stylometry = torch.tensor(self.manual_stylomtries[idx], dtype=torch.float)        

        if author_score is not None:
            # Non-testing
            return (combined_text, author_score, style_similarity_score, manual_stylometry, author_score)
        else:
            # Testing
            return (combined_text, style_similarity_score, manual_stylometry)


### Original AVDataset
- Does not support stylometric features
- Only used to generate customised input csv files that contain ```style_similarity_score```
- #### Columns of ```csv_file```
    - ```text_1```
    - ```text_2```
    - ```label```

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class OriginalAVDataset(Dataset):
    def __init__(self, csv_file: str, text_1_column="text_1", text_2_column="text_2", separation_token="[SEP]"):
        # Load CSV
        self.df = pd.read_csv(csv_file)

        # Columns
        self.text_1 = self.df[text_1_column].astype(str).tolist()
        self.text_2 = self.df[text_2_column].astype(str).tolist()

        self.separation_token = separation_token

        # Sanity check
        assert len(self.text_1) == len(self.text_2), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)}"

    def __len__(self):
        return len(self.text_1)

    def __getitem__(self, idx):
        # Current: Return a tuple instead
        combined_text = (self.text_1[idx], self.text_2[idx])

        return combined_text


### MultiHeadCrossEncoder

In [ ]:
import torch
from torch import nn, functional as F
from transformers import AutoModel

"""
Return different scores, which will be used to determine the final score in the task of Author Verification
"""


class MultiHeadCrossEncoder(nn.Module):
    model_name: str
    head_weights: torch.Tensor

    def __init__(self, model_name: str, head_weights: torch.Tensor, manual_stylometry_dim: int = 171):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        encoder_hidden_size = self.encoder.config.hidden_size

        self.shared = nn.Linear(encoder_hidden_size, encoder_hidden_size)
        self.shared_activation = nn.ReLU()

        self.head_author = nn.Linear(encoder_hidden_size, 1)
        self.head_style_similarity = nn.Linear(encoder_hidden_size, 1)
        self.head_manual_stylometry = nn.Sequential(
            nn.Linear(manual_stylometry_dim, 64), 
            nn.ReLU(), 
            nn.Linear(64, 1)
        )

        assert isinstance(
            head_weights, torch.Tensor), f"Invalid type of head_weights: {type(head_weights)}"
        assert head_weights.dim() == 1 and head_weights.numel() == 3, f"Invalid shape of head_weights (expect [3]): {head_weights.shape}"

        if sum(head_weights) == 1:
            self.head_weights = head_weights
        else:
            # Apply softmax function
            self.head_weights = F.softmax(head_weights, dim=0)

    def forward(self, input_ids, attention_mask, manual_stylometry_vector: torch.Tensor, token_type_ids=None):
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        embedding_cls = encoder_outputs.last_hidden_state[:, 0]

        shared_output = self.shared_activation(self.shared(embedding_cls))

        # Individual heads
        score_author = self.head_author(shared_output)
        score_style_similarity = self.head_style_similarity(shared_output)
        score_manual_stylometry = self.head_manual_stylometry(manual_stylometry_vector)

        score_final = (self.head_weights[0] * score_author) + (self.head_weights[1] * score_style_similarity) + (self.head_weights[2] * score_manual_stylometry)

        return {
            f"score_final": score_final,
            f"score_author": score_author,
            f"score_style_similarity": score_style_similarity, 
            f"score_manual_stylometry": score_manual_stylometry
        }


## Utils

In [ ]:
def log(text=None) -> None:
    if text is None:
        print(f"[INFO]")
    else:
        print(f"[INFO] {text}")

## Global Variable

In [ ]:
# TODO: Update me if applicable
INPUT_FOLDER_PATH = os.path.join("/", "kaggle", "input", "datasets", "w45242hy", "authorverification")
# TODO: Update me if applicable
# TODO: Make it the same as INPUT_FOLDER_PATH
OUTPUT_FOLDER_PATH = os.path.join("/", "kaggle", "working")

# Input CSV file
# TODO: Update me if applicable
CSV_NAME = f"test.csv"
CSV_PATH = os.path.join(INPUT_FOLDER_PATH, CSV_NAME)

# Output CSV file
# TODO: Update me if applicable
CSV_NAME_OUTPUT = f"predictions.csv"
CSV_PATH_OUTPUT = os.path.join(OUTPUT_FOLDER_PATH, CSV_NAME_OUTPUT)

# ============================================ Update variables above ===================================================

# If prediction >= SIGMOID_THRESHOLD, then it is set to 1. 0 otherwise. 
SIGMOID_THRESHOLD = 0.5
SEPARATION_TOKEN = f"[SEP]"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"Using device {device} ...")
# Larger batch for efficiency
BATCH_SIZE = 32
SHUFFLE_DATA = False
TRAIN_EPOCH = 1
LEARNING_RATE = 2e-5
# Number of train_epoch to log once
EPOCH_PER_LOG = 1

HEAD_WEIGHTS = torch.tensor([0.8, 0.1, 0.1], dtype=torch.float)
MODEL_NAME = f"bert-base-cased"
# NOTE: Hard-coded
# CROSS_ENCODER_NAME = f"cross-encoder_{HEAD_WEIGHTS.tolist()}_{MODEL_NAME}.pth"
CROSS_ENCODER_NAME = f"cross-encoder_080101_{MODEL_NAME}.pth"
CROSS_ENCODER_PATH = os.path.join(OUTPUT_FOLDER_PATH, CROSS_ENCODER_NAME)

CSV_NAME_TRAIN = f"train.csv"
CSV_PATH_TRAIN = os.path.join(INPUT_FOLDER_PATH, CSV_NAME_TRAIN)

# For generating csv file of style similarity
OUTPUT_CSV_STYLE_SIMILARITY_NAME = f"{CSV_NAME.split('.')[0]}_final.{CSV_NAME.split('.')[1]}"
OUTPUT_CSV_STYLE_SIMILARITY_PATH = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_CSV_STYLE_SIMILARITY_NAME)
MODEL_NAME_STYLE_SIMILARITY = f"sentence-transformers/all-MiniLM-L6-v2"

# For generating csv file of stylometric features
nlp = spacy.load("en_core_web_md", disable=["ner"])
MAX_LEN_STYLOMETRIC_FEATURE = 256
BATCH_SIZE_STYLOMETRIC_FEATURE = 64            # For two GPUs (32 each)
OUTPUT_CSV_STYLOMETRIC_FEATURES_NAME = f"{CSV_NAME.split('.')[0]}_stylometric_features.{CSV_NAME.split('.')[1]}"
OUTPUT_CSV_STYLOMETRIC_FEATURES_PATH = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_CSV_STYLOMETRIC_FEATURES_NAME)

print(f"Global variables are set ...")

## Generate Customised Input Files
- ### Style Similarity Score
- ### Stylometric Features

### Stylometric Feature

In [ ]:
# makes sure that these are lowercase
TOP_FUNCTION_WORDS = set(word.lower() for word in [
    "the", "of", "and", "to", "a", "in", 
    "is", "you", "that", "was", "he", "his", 
    "for", "it", "with", "as", "on", "are",
    "be", "at", "by", "they", "this", "from", 
    "i", "have", "or", "had", "not", "she"
])
SORTED_TOP_FUNCTION_WORDS = sorted(list(TOP_FUNCTION_WORDS))

FUNCTION_TAGS = {"ADP", "AUX", "CONJ", "CCONJ", "DET", "PART", "PRON", "SCONJ"}
CONTENT_TAGS = {"ADJ", "ADV", "INTJ", "NOUN", "NUM", "PROPN", "VERB"}

NON_WORD_TAGS = {"NUM", "PUNCT", "SYM", "X"}

ALL_NN_FORMS = {"NN", "NNS", "NNP", "NNPS"}                # all noun forms
ALL_JJ_FORMS = {"JJ", "JJR", "JJS"}                        # all adjective forms
ALL_RB_FORMS = {"RB", "RBR", "RBS"}                        # all adverb forms
ALL_VB_FORMS = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}   # all verb forms

def get_top_pos_trigrams():
    trigrams = set()

    for nn in ALL_NN_FORMS:
        trigrams.add((nn, "IN", "DT"))
        trigrams.add(("IN", "DT", nn))
        trigrams.add(("DT", nn, "IN"))
        trigrams.add(("DT", nn, "PUNCT"))
        for nn2 in ALL_NN_FORMS:
            trigrams.add(("DT", nn, nn2))

    for jj in ALL_JJ_FORMS:
        trigrams.add(("IN", "DT", jj))
        for nn in ALL_NN_FORMS:
            trigrams.add(("DT", jj, nn))
            trigrams.add((jj, nn, "IN"))
            trigrams.add((jj, nn, "PUNCT"))
            for rb in ALL_RB_FORMS:
                trigrams.add((rb, jj, nn))

    for vb in ALL_VB_FORMS:
        trigrams.add(("PRP", vb, "DT"))
        trigrams.add(("PRP", vb, "IN"))

    return trigrams
                
TOP_POS_TRIGRAMS = get_top_pos_trigrams()
SORTED_TOP_POS_TRIGRAMS = sorted(list(TOP_POS_TRIGRAMS))
# print(len(TOP_POS_TRIGRAMS))


def extract_stylometric_features(document):
    """
    Processes a spaCy document and returns a []-dimensional feature vector
    """
    f_count, c_count = 0, 0                          # counts of function and content words
    top_function_word_freqs = defaultdict(int)       # frequencies of top function words
    lexical_count = 0                                # count of all words
    unique_words = set()                             # set of all words (no duplicates)
    conjunction_count = 0                            # count of all conjunction words
    long_word_count = 0                              # long word (>6 chars) count
    oov_count, non_oov_count = 0, 0                  # counts of oov and vocabulary words
    word_length_sum = 0                              # sum of all word lengths
    pronoun_freqs = {                                # frequencies of various pronoun types
        "1S": 0,                             # first-person singular
        "1P": 0,                             # first-person plural
        "12": 0,                             # first- and second-person
        "3": 0                               # third-person
    }
    special_punct_freqs = defaultdict(int)           # frequencies of each of ";,.-?!"
    total_punct_count = 0                            # total punctuation count
    top_pos_trigram_freqs = defaultdict(int)         # frequencies of the top POS trigrams
    total_pos_trigram_count = 0                      # total POS trigram count

    # loop over all tokens
    for token in document:
        if token.is_space:
            continue

        text_lower = token.text.lower()
        universal_pos = token.pos_

        # word statistics
        if universal_pos not in NON_WORD_TAGS:
            unique_words.add(text_lower)
            lexical_count += 1
            word_length_sum += len(token.text)

            if text_lower in TOP_FUNCTION_WORDS:
                top_function_word_freqs[text_lower] += 1

            if universal_pos in FUNCTION_TAGS:
                f_count += 1
            elif universal_pos in CONTENT_TAGS:
                c_count += 1

            if universal_pos in ["CONJ", "CCONJ"]:
                conjunction_count += 1
            
            if len(token.text) > 6:
                long_word_count += 1

            if token.is_oov:
                oov_count += 1
            else:
                non_oov_count += 1

        # punctuation statistics
        if universal_pos == "PUNCT":
            total_punct_count += 1
            if token.text in ";,.-?!":
                special_punct_freqs[token.text] += 1

        # personal pronoun statistics
        if universal_pos == "PRON" and "Prs" in token.morph.get("PronType"):
            person = token.morph.get("Person")
            num = token.morph.get("Number")
            if "1" in person:
                pronoun_freqs["12"] += 1
                if "Sing" in num:
                    pronoun_freqs["1S"] += 1
                else:
                    pronoun_freqs["1P"] += 1
            elif "2" in person:
                pronoun_freqs["12"] += 1
            elif "3" in person:
                pronoun_freqs["3"] += 1

    # extract Penn Treebank POS tags
    ptb_tags = [token.tag_ for token in document]

    # loop over all POS trigrams
    for tri in ngrams(ptb_tags, 3):
        total_pos_trigram_count += 1
        if tri in TOP_POS_TRIGRAMS:
            top_pos_trigram_freqs[tri] += 1

    # sentence statistics
    sentence_lengths = [len([t for t in s if not t.is_space]) for s in document.sents]
    mean_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0
    if len(sentence_lengths) > 1:
        variance = sum((length - mean_sentence_length)**2 for length in sentence_lengths) / len(sentence_lengths)
        std_sentence_length = math.sqrt(variance)
    else:
        std_sentence_length = 0

    # final feature assembly
    feature_vector = []

    # the (normalised) frequencies of each of the top 30 function words [30 features]
    for word in SORTED_TOP_FUNCTION_WORDS:
        feature_vector.append(top_function_word_freqs[word] / (f_count + 1e-9))

    # the (normalised) ratio of function words to content words [1 feature]
    feature_vector.append(np.clip(f_count / (c_count + 1e-9), 0, 5))

    # the (normalised) type-token ratio (i.e proportion of non-unique words) [1 feature]
    feature_vector.append(len(unique_words) / (lexical_count + 1e-9))

    # the (normalised) count of conjunctions (e.g. and, but) [1 feature]
    feature_vector.append(conjunction_count / (f_count + 1e-9))

    # the (normalised) count of long words (>6 chars) [1 feature]
    feature_vector.append(long_word_count / (lexical_count + 1e-9))

    # the (normalised) ratio of OOV words to non-OOV words [1 feature]
    feature_vector.append(np.clip(oov_count / (non_oov_count + 1e-9), 0, 5))

    # the (normalised) ratio of first- and second-person pronouns to third-person pronouns [1 feature]
    feature_vector.append(np.clip(pronoun_freqs["12"] / (pronoun_freqs["3"] + 1e-9), 0, 5))

    # the (normalised) ratio of first-person singular pronouns to first-person plural pronouns [1 feature]
    feature_vector.append(np.clip(pronoun_freqs["1S"] / (pronoun_freqs["1P"] + 1e-9), 0, 5))

    # the mean word length [1 feature]
    feature_vector.append(np.clip(word_length_sum / (lexical_count + 1e-9), 0, 20))

    # the mean sentence length [1 feature]
    feature_vector.append(np.clip(mean_sentence_length, 0, 100))

    # the sentence length standard deviation [1 feature]
    feature_vector.append(np.clip(std_sentence_length, 0, 100))

    # the (normalised) frequencies of punctuation symbols in ';,.-?!' [6 features]
    for symbol in ";,.-?!":
        feature_vector.append(special_punct_freqs[symbol] / (total_punct_count + 1e-9))

    # the (normalised) ratios of punctuation symbol pairs of interest [6 features]
    punctuation_pairs = [
        (",", ";"), (",", "-"),
        (";", "-"), ("?", "."),
        (",", "."), ("!", ".")
    ]
    for s1, s2 in punctuation_pairs:
        feature_vector.append(np.clip(special_punct_freqs[s1] / (special_punct_freqs[s2] + 1e-9), 0, 5))

    # the (normalised) counts of the top POS trigrams [119 features]
    for tri in SORTED_TOP_POS_TRIGRAMS:
        feature_vector.append(top_pos_trigram_freqs[tri] / (total_pos_trigram_count + 1e-9))

    return np.array(feature_vector)

def process_input_stylometry(df):
    feature_vectors_1 = []
    feature_vectors_2 = []

    for document in nlp.pipe(df["text_1"], batch_size=256, n_process=-1, disable=["ner", "lemmatizer"]):
        feature_vectors_1.append(extract_stylometric_features(document))

    for document in nlp.pipe(df["text_2"], batch_size=256, n_process=-1, disable=["ner", "lemmatizer"]):
        feature_vectors_2.append(extract_stylometric_features(document))
        
    return np.array(feature_vectors_1), np.array(feature_vectors_2)

In [ ]:
print(f"Reading csv file at {CSV_PATH} ...")

train_df_stylometric_features = pd.read_csv(CSV_PATH_TRAIN, encoding="utf-8")
target_df_stylometric_features = pd.read_csv(CSV_PATH, encoding="utf-8")

train_style_1, train_style_2 = process_input_stylometry(train_df_stylometric_features)
target_style_1, target_style_2 = process_input_stylometry(target_df_stylometric_features)

# Different features can have different ranges, so we need to scale them

combined_stylometric_features_train = np.vstack([train_style_1, train_style_2])
combined_stylometric_features_target = np.vstack([target_style_1, target_style_2])

scaler = StandardScaler().fit(combined_stylometric_features_train)

# Scale all features
target_style_scaled_1 = scaler.transform(target_style_1)
target_style_scaled_2 = scaler.transform(target_style_2)

train_style_scaled_1 = scaler.transform(train_style_1)
train_style_scaled_2 = scaler.transform(train_style_2)

# Compute the difference
train_style_diff = np.abs(train_style_scaled_1 - train_style_scaled_2)
target_style_diff = np.abs(target_style_scaled_1 - target_style_scaled_2)

print(f"train_style_diff: {train_style_diff.shape}")
print(f"target_style_diff: {target_style_diff.shape}")

train_df = pd.DataFrame(train_style_diff)
target_df = pd.DataFrame(target_style_diff)

target_df.to_csv(OUTPUT_CSV_STYLOMETRIC_FEATURES_PATH, index=False)

print(f"target_df is saved to {OUTPUT_CSV_STYLOMETRIC_FEATURES_PATH} ...")

## Style Similarity Score

In [ ]:
print(f"Reading csv file at {CSV_PATH} ...")
    
# Load dataset
dataset_raw = OriginalAVDataset(CSV_PATH)
dataloader_raw = DataLoader(dataset_raw, batch_size=8, shuffle=False)  # Larger batch for efficiency

# Load pre-trained sentence transformer
model = SentenceTransformer(MODEL_NAME_STYLE_SIMILARITY)

style_similarity_scores = []
all_text_1 = []
all_text_2 = []

for (texts_1, texts_2) in dataloader_raw:
    # (texts_1, texts_2) is 1 batch

    # Compute embeddings
    # Shape: [batch_size, feature_num]
    embedding_1 = model.encode(texts_1, convert_to_tensor=True)
    # Shape: [batch_size, feature_num]
    embedding_2 = model.encode(texts_2, convert_to_tensor=True)

    # Cosine similarity
    # Shape: [batch_size]
    style_similarity_score = torch.nn.functional.cosine_similarity(embedding_1, embedding_2, dim=1)

    # Save data
    style_similarity_scores += style_similarity_score.detach().cpu().tolist()
    all_text_1 += list(texts_1)
    all_text_2 += list(texts_2)

# Create new DataFrame
df_style_similarity = pd.DataFrame({
    "text_1": all_text_1,
    "text_2": all_text_2,
    "style_similarity_score": style_similarity_scores
})

# Save CSV
df_style_similarity.to_csv(OUTPUT_CSV_STYLE_SIMILARITY_PATH, index=False)
print(f"Final training data saved to {OUTPUT_CSV_STYLE_SIMILARITY_PATH} ...")

#### All input data have been prepared

## Prepare Data

In [ ]:
log(f"Reading csv files at {OUTPUT_CSV_STYLE_SIMILARITY_PATH} and {OUTPUT_CSV_STYLOMETRIC_FEATURES_PATH} ...")

dataset_test = AVDataset(OUTPUT_CSV_STYLE_SIMILARITY_PATH, OUTPUT_CSV_STYLOMETRIC_FEATURES_PATH)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)

print(f"All data finish preparing ...")

## Instantiate Entity

In [ ]:
# Instantiate relevant models
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
cross_encoder = MultiHeadCrossEncoder(model_name=MODEL_NAME, head_weights=HEAD_WEIGHTS).to(device)
cross_encoder.load_state_dict(torch.load(CROSS_ENCODER_PATH))
log(f"Cross encoder is loaded from {CROSS_ENCODER_PATH} ...")
# This expects raw numbers (logits) and handles the Sigmoid internally
criterion = nn.BCEWithLogitsLoss()

print(f"All instances have been instantiated ...")

## Testing

In [ ]:
cross_encoder.eval()

# Will be saved in predictions.csv
predictions = []

with torch.no_grad():
    for batch_X, batch_style_similarity_score, batch_manual_stylometry_X in dataloader_test:
        batch_style_similarity_score = batch_style_similarity_score.to(device)
        batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)

        """
        {
            "input_ids": tensor, 
            "token_type_ids": tensor, 
            "attention_mask": tensor
        }
        """
        encoding_X = tokenizer(
            batch_X[0], 
            batch_X[1], 
            padding=True, 
            truncation=True, 
            return_tensors=f"pt"
        ).to(device)

        y_predictions = cross_encoder(
            input_ids=encoding_X["input_ids"], 
            attention_mask=encoding_X["attention_mask"],
            token_type_ids=encoding_X.get("token_type_ids"),
            manual_stylometry_vector=batch_manual_stylometry_X
        )

        # Shape: [batch_size]
        y_prediction_final = (torch.sigmoid(y_predictions["score_final"].detach().view(-1)) >= SIGMOID_THRESHOLD).int()

        predictions.extend(y_prediction_final.tolist())

log(f"Finish generating predictions on the testing dataset ...")

### Generate Prediction CSV File

In [ ]:
df_predictions = pd.DataFrame(predictions, columns=["prediction"])

df_predictions.to_csv(CSV_PATH_OUTPUT, index=False)

print(f"Predictions are saved to {CSV_PATH_OUTPUT} ...")